# CIFAR-10 Dynamic Robustness Benchmark

This notebook is the single orchestration entry point for the project. The implementation lives in Python modules; this notebook checks the environment, lists registered models, and launches the final adaptive dynamic robustness pipeline.

## 1. Environment Check

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Model Registry

In [ ]:
from Models import MODEL_REGISTRY, models_by_group

for name, config in MODEL_REGISTRY.items():
    print(
        f"{name:22s} | {config.capacity_level:18s} | "
        f"{config.design_type:19s} | batch={config.default_batch_size:3d} | "
        f"epochs={config.default_epochs:3d} | lr={config.default_lr}"
    )

## 3. Benchmark Configuration

`STAGE = "all"` runs calibration, protocol selection, final experiments, summary figures, and path-dependent phase analysis. For a faster continuation run, set `RESUME = True`.

In [ ]:
GROUP = "all"
STAGE = "all"
RESUME = False
TRAIN_SUBSET = 0
CALIBRATION_EVAL_SUBSET = 2000
FINAL_EVAL_SUBSET = 0
NUM_WORKERS = 0
SEED = 42

config = {
    "group": GROUP,
    "stage": STAGE,
    "resume": RESUME,
    "train_subset": TRAIN_SUBSET,
    "calibration_eval_subset": CALIBRATION_EVAL_SUBSET,
    "final_eval_subset": FINAL_EVAL_SUBSET,
    "num_workers": NUM_WORKERS,
    "seed": SEED,
    "models": models_by_group(GROUP),
}
print(json.dumps(config, indent=2))

## 4. Run Adaptive Dynamic Robustness Pipeline

In [ ]:
cmd = [
    sys.executable,
    "-u",
    "Experiments/run_adaptive_protocol.py",
    "--stage", STAGE,
    "--group", GROUP,
    "--train-subset", str(TRAIN_SUBSET),
    "--calibration-eval-subset", str(CALIBRATION_EVAL_SUBSET),
    "--final-eval-subset", str(FINAL_EVAL_SUBSET),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
]
if RESUME:
    cmd.append("--resume")

print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

## 5. Final Output Locations

In [ ]:
final_paths = [
    RESULTS_DIR / "adaptive_protocol" / "summary_figures",
    RESULTS_DIR / "adaptive_protocol" / "path_dependent_phase",
    RESULTS_DIR / "adaptive_protocol" / "path_dependent_phase" / "PATH_DEPENDENT_PHASE_SUMMARY.md",
    RESULTS_DIR / "adaptive_protocol" / "path_dependent_phase" / "path_dependent_phase_timeline.png",
]

for path in final_paths:
    print(path, "exists=" + str(path.exists()))